In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

class DataAnalyzer:
    # --- Initialization --- 
    def __init__(self, file_path=None):
        self.df = pd.DataFrame()  # To store dataframe
        self.numpy_array = np.array([])  # To store Numpy array
        self.file_path = file_path  # To store file 
        self.last_plot = None  # To store the last generated figure object
        if file_path:
            self.load_data(file_path)

    # --- Destructor ---
    def __del__(self):
        print("\nAnalyzer object destroyed. Cleanup completed.")
    
    # --- Load CSV file ---
    def load_data(self, file_path=None):
        if not file_path:
            file_path = input("Enter CSV file path: ").strip()
        try:
            self.df = pd.read_csv(file_path)
            self.numpy_array = self.df.to_numpy()  # convert dataframe into a Numpy array
            print("\n File loaded successfully!")
        except FileNotFoundError:
            print("\n File not found.")
        except Exception as e:
            print(f"\n Error: {e}")

    # --- Explore data ---
    def explore(self):
        if self.df.empty:
            print(" Please load data first.")
            return

        while True:
            print("\n=== Explore Data ===")
            print("1. Display First 5 Rows")
            print("2. Display Last 5 Rows")
            print("3. Display Column Names")
            print("4. Display Data Types")
            print("5. Display Basic Info")
            print("6. Go Back")
            choice = input("Enter your choice: ").strip()

            if choice == "1":  print(self.df.head())
            elif choice == "2": print(self.df.tail())
            elif choice == "3": print(list(self.df.columns))
            elif choice == "4":  print(self.df.dtypes)
            elif choice == "5": self.df.info()
            elif choice == "6":  break
            else: print("Invalid choice!")

    # --- Handle missing values --- 
    def clean_data(self):
        if self.df.empty:
            print(" Please load the data first.")
            return
    
        print("\n=== Data Cleaning Started ===")
        rows_before = len(self.df)
        missing_before = self.df.isnull().sum().sum()
    
        # Step 1: Remove duplicate rows
        duplicates = self.df.duplicated().sum()
        if duplicates > 0:
            self.df.drop_duplicates(inplace=True)
            print(f" Removed {duplicates} duplicate rows.")
    
        # Step 2: Drop rows with too many missing values
        threshold = int(len(self.df.columns) * 0.5)  # drop rows with >50% missing values
        rows_dropped = self.df[self.df.isnull().sum(axis=1) > threshold].shape[0]
        if rows_dropped > 0:
            self.df.dropna(thresh=threshold, inplace=True)
            print(f" Dropped {rows_dropped} rows with >50% missing values.")
    
        # Step 3: Handle missing values column-wise
        for col in self.df.columns:
            if self.df[col].isnull().sum() == 0:
                continue  # no missing values in this column
    
            if pd.api.types.is_numeric_dtype(self.df[col]):
                mean_val = self.df[col].mean()
                self.df[col].fillna(mean_val, inplace=True)
                print(f" Filled missing numeric values in '{col}' with mean ({mean_val:.2f}).")
            else:
                self.df[col].fillna("Unknown", inplace=True)
                print(f" Filled missing categorical values in '{col}' with 'Unknown'.")
    
        # Step 4: Show cleaning summary
        rows_after = len(self.df)
        missing_after = self.df.isnull().sum().sum()
    
        print("\n Cleaning Completed:")
        print(f" - Rows before: {rows_before}")
        print(f" - Rows after: {rows_after}")
        print(f" - Missing values before: {missing_before}")
        print(f" - Missing values after: {missing_after}")
        print("==============================")


    # --- Convert to numpy ---
    def convert_numpy(self):
        if self.df.empty:
            print(" Load data first.")
            return
    
        try:
            cols = input("Enter column names to convert to NumPy (comma-separated, or leave blank for all): ").strip()
            if cols:
                cols = [c.strip() for c in cols.split(',')]
                self.numpy_array = self.df[cols].values
            else:
                self.numpy_array = self.df.values
            print("\n Data successfully converted into a NumPy array.") 
    
            row_index = input("Enter row index to access (leave blank to skip): ").strip()
            col_index = input("Enter column index to access (leave blank to skip): ").strip()
            if row_index.isdigit() and col_index.isdigit():
                row_index, col_index = int(row_index), int(col_index)
                print(f"Value at row {row_index}, column {col_index}: {self.numpy_array[row_index, col_index]}")
            else:
                print("Skipping indexing.")
    
            # --- slicing ---
            def parse_slice(s):
                if not s:
                    return slice(None)
                if ':' in s:
                    start, end = s.split(':')
                    start = int(start.strip()) if start.strip() else None
                    end = int(end.strip()) if end.strip() else None
                    return slice(start, end)
                return int(s)
    
            row_slice = parse_slice(input("Enter row slice (start:end, leave blank for all): ").strip())
            col_slice = parse_slice(input("Enter column slice (start:end, leave blank for all): ").strip())
    
            print(f"\nSliced array:\n{self.numpy_array[row_slice, col_slice]}")
    
        except KeyError:
            print(" Columns not found in the dataset.")
        except Exception as e:
            print(f" Error: {e}")

    # --- Mathematical operations ---
    def mathematical_operations(self):
        if self.df.empty:
            print(" Please load a dataset first.")
            return
    
        # Step 1: Identify numeric columns
        numeric_cols = self.df.select_dtypes(include=np.number).columns.tolist()
        if not numeric_cols:
            print(" No numeric columns found to perform operations.")
            return
    
        print("\nNumeric columns available:", numeric_cols)
    
        # Step 2: Choose operation type
        print("\nAvailable operations:")
        print("1. Add two columns")
        print("2. Subtract two columns")
        print("3. Multiply two columns")
        print("4. Divide two columns")
        print("5. Percentage of one column over another")
        print("6. Compute mean or sum of selected columns")
        print("7. Back to main menu")
    
        op = input("Enter operation number: ").strip()
    
        if op == "7":
            return
    
        # Step 3: Select columns for operation
        if op in ["1", "2", "3", "4", "5"]:
            col1 = input("Enter first numeric column: ").strip()
            col2 = input("Enter second numeric column: ").strip()
            if col1 not in numeric_cols or col2 not in numeric_cols:
                print(" Invalid columns selected.")
                return
            new_col_name = input("Enter new column name for result: ").strip()
    
            if op == "1":
                self.df[new_col_name] = self.df[col1] + self.df[col2]
                print(f" {new_col_name} = {col1} + {col2}")
    
            elif op == "2":
                self.df[new_col_name] = self.df[col1] - self.df[col2]
                print(f" {new_col_name} = {col1} - {col2}")
    
            elif op == "3":
                self.df[new_col_name] = self.df[col1] * self.df[col2]
                print(f" {new_col_name} = {col1} * {col2}")
    
            elif op == "4":
                self.df[new_col_name] = self.df[col1] / self.df[col2].replace(0, np.nan)
                print(f" {new_col_name} = {col1} / {col2} (division by zero replaced with NaN)")
    
            elif op == "5":
                self.df[new_col_name] = (self.df[col1] / self.df[col2].replace(0, np.nan)) * 100
                print(f" {new_col_name} = {col1} / {col2} * 100 (%)")
    
        elif op == "6":
            selected_cols = input("Enter numeric columns to aggregate (comma-separated): ").strip().split(',')
            selected_cols = [c.strip() for c in selected_cols if c.strip() in numeric_cols]
            if not selected_cols:
                print(" No valid numeric columns selected.")
                return
            agg_func = input("Enter aggregation (sum / mean): ").strip().lower()
            if agg_func == "sum":
                self.df["Sum"] = self.df[selected_cols].sum(axis=1)
                print(f" Sum column created for {selected_cols}")
            elif agg_func == "mean":
                self.df["Mean"] = self.df[selected_cols].mean(axis=1)
                print(f" Mean column created for {selected_cols}")
            else:
                print(" Invalid aggregation function.")
    
        else:
            print(" Invalid operation selected.")
    
        # Step 4: Show first 5 rows of new columns
        print("\nFirst 5 rows after operation:")
        print(self.df.head())


    # --- Combine data ---
    def combine_data(self):
        file_path = input("Enter CSV path of dataset to combine: ").strip()
        try:
            other_df = pd.read_csv(file_path)
            self.df = pd.concat([self.df, other_df], ignore_index=True)
            print(f" Datasets combined. Total rows: {len(self.df)}")
        except Exception as e:
            print(f" Error: {e}")

    # --- Split data ---
    def split_data(self):
        if self.df.empty:
            print(" Load data first.")
            return

        col = input("Enter column name to split by: ").strip()
        if col not in self.df.columns:
            print(f" Column '{col}' not found.")
            return
            
        val = input(f"Enter value of '{col}' to filter first dataset: ").strip()
        if pd.api.types.is_numeric_dtype(self.df[col]):
            try:
                val = float(val)
            except ValueError:
                print(f"Invalid numeric value '{val}' for column '{col}'.")
                return

        df1 = self.df[self.df[col] == val].copy()
        df2 = self.df[self.df[col] != val].copy()
        print(f" Dataset 1 ('{col}' == {val}) rows: {len(df1)}")
        print(f" Dataset 2 ('{col}' != {val}) rows: {len(df2)}")
        return df1, df2

    # --- Search, Sort, Filter ---
    def search_sort_filter(self):
        if self.df.empty:
            print(" Load data first.")
            return
 
        col = input("Enter column name to search: ").strip()
        if col not in self.df.columns:
            print(" Invalid column.")
            return

        val = input(f"Enter value to search in '{col}': ").strip()
        results = self.df[self.df[col].astype(str).str.contains(val, case=False, na=False)]
        print(f"\nFound {len(results)} rows where '{col}' contains '{val}':")
        print(results.head())

        sort_col = input("\nEnter column name to sort by: ").strip()
        if sort_col in self.df.columns:
            sorted_df = self.df.sort_values(by=sort_col, ascending=False)
            print(f"\nTop 5 rows sorted by '{sort_col}' (descending):")
            print(sorted_df.head())
        else:
            print(f" Column '{sort_col}' not found.")

        filter_col = input("\nEnter numeric column to filter by: ").strip()
        if filter_col in self.df.columns:
            try:
                filter_val = float(input(f"Enter minimum value for '{filter_col}': ").strip())
                filtered_df = self.df[self.df[filter_col] >= filter_val]
                print(f"\nFiltered rows where '{filter_col}' >= {filter_val}:")
                print(filtered_df.head())
            except ValueError:
                print(" Invalid numeric input.")
        else:
            print(f" Column '{filter_col}' not found.")

    # --- Aggregate functions ---
    def aggregate_functions(self):
        if self.df.empty:
            print(" Load data first.")
            return

        col = input("Enter column name to aggregate (e.g., Sales): ").strip()
        if col not in self.df.columns or not pd.api.types.is_numeric_dtype(self.df[col]):
            print(f" Column '{col}' must be numeric.")
            return

        print(f"\nAggregate Functions for '{col}':")
        print(f"Sum: {self.df[col].sum():.2f}")
        print(f"Mean: {self.df[col].mean():.2f}")
        print(f"Count: {self.df[col].count()}")

    # --- Statistical analysis ---
    def statistical_analysis(self):
        if self.df.empty:
            print(" Load data first.")
            return

        col = input("Enter column name for statistical analysis: ").strip()
        if col not in self.df.columns or not pd.api.types.is_numeric_dtype(self.df[col]):
            print(f" Column '{col}' must be numeric.")
            return

        print(f"\nStatistical Analysis for '{col}':")
        print(f"Standard Deviation: {self.df[col].std():.2f}")
        print(f"Variance: {self.df[col].var():.2f}")
        print("Quantiles:")
        print(self.df[col].quantile([0.25, 0.5, 0.75]))

    # --- Pivot table ---
    def create_pivot_table(self):
        if self.df.empty:
            print(" Load data first.")
            return
        
        try:
            index_col = [c.strip() for c in input("Enter index columns (comma-separated): ").split(',') if c.strip()]
            values_col = [c.strip() for c in input("Enter value columns (comma-separated): ").split(',') if c.strip()]
            agg_func = input("Enter aggregation function (sum, mean, count): ").strip()

            if not index_col or not values_col:
                print(" Invalid input — please enter valid column names.")
                return

            pivot = pd.pivot_table(self.df, values=values_col, index=index_col, aggfunc=agg_func)
            print("\n Pivot Table:")
            print(pivot)
        except Exception as e:
            print(f" Error: {e}")

    # --- Visualization ---
    def visualize_data(self):
        if self.df.empty:
            print(" Load data first.")
            return
        
        sns.set_theme(style="whitegrid")
        
        while True:
            print("\n==== Visualization Menu ====")
            print("1. Bar Plot")
            print("2. Box Plot")
            print("3. Scatter Plot")
            print("4. Histogram")
            print("5. Heatmap")
            print("6. Pie Chart")
            print("7. Stack Plot")
            print("8. Go Back")
            choice = input("Enter plot type number: ").strip()

            if choice == "8":
                break
            elif choice not in map(str, range(1,8)):
                print("Invalid choice.")
                continue

            try:
                self.last_plot = plt.figure(figsize=(10, 6))
                
                if choice == "1":
                    x_col = input("Enter categorical column for X-axis: ").strip()
                    y_col = input("Enter numeric column for Y-axis: ").strip()
                    sns.barplot(data=self.df, x=x_col, y=y_col, estimator=sum, errorbar=None, palette='viridis', hue=x_col, legend=False)
                    plt.title(f'Total {y_col} by {x_col}')
                    plt.xticks(rotation=45)
        
                elif choice == "2":
                    x_col = input("Enter categorical column for X-axis: ").strip()
                    y_col = input("Enter numeric column for Y-axis: ").strip()
                    sns.boxplot(data=self.df, x=x_col, y=y_col, hue=x_col, palette='pastel', legend=False)
                    plt.title(f'Distribution of {y_col} by {x_col}')
                    plt.xticks(rotation=45)
                    
                elif choice == "3":
                    x_col = input("Enter X-axis column: ").strip()
                    y_col = input("Enter Y-axis column: ").strip()
                    hue_col = input("Enter column for color (optional): ").strip()
                    sns.scatterplot(data=self.df, x=x_col, y=y_col, hue=hue_col if hue_col else None, palette='deep', alpha=0.7)
                    plt.title(f'{y_col} vs {x_col}')
        
                elif choice == "4":
                    col = input("Enter numeric column for histogram: ").strip()
                    bins = int(input("Enter number of bins (e.g., 30): "))
                    sns.histplot(self.df[col], bins=bins, kde=True, color='purple')
                    plt.title(f'Distribution of {col}')
                
                elif choice == "5":
                    numeric_df = self.df.select_dtypes(include=np.number)
                    corr_matrix = numeric_df.corr()
                    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
                    plt.title('Correlation Matrix of Numeric Columns')

                elif choice == "6":
                    col = input("Enter column for pie chart: ").strip()
                    counts = self.df[col].value_counts().nlargest(10)
                    plt.pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=140)
                    plt.title(f'Distribution of {col}')
                    plt.axis('equal')

                elif choice == "7":
                    time_col = input("Enter time/date column for X-axis: ").strip()
                    category_col = input("Enter category column to stack: ").strip()
                    value_col = input("Enter numeric column for values: ").strip()

                    pivot_data = self.df.pivot_table(index=time_col, columns=category_col, values=value_col, aggfunc='sum').fillna(0)
                    
                    if pivot_data.empty:
                        print(" Could not create pivot table for stack plot.")
                        plt.close(self.last_plot)
                        self.last_plot = None
                        continue

                    plt.stackplot(pivot_data.index, pivot_data.values.T, labels=pivot_data.columns, alpha=0.8)
                    plt.title(f'{value_col} Stacked by {category_col} Over {time_col}')
                    plt.xlabel(time_col)
                    plt.ylabel(f"Total {value_col}")
                    plt.legend(title=category_col, loc='upper left')
                    plt.xticks(rotation=45)

                self.last_plot.tight_layout()
                print("\n Plot generated. Use option 7 in main menu to save it.")

            except Exception as e:
                print(f" Error: {e}")
                if self.last_plot:
                    plt.close(self.last_plot)
                self.last_plot = None

    # --- Save visualization ---
    def save_visualization(self):
        if self.last_plot is None:
            print("\n No plot generated yet. Please create a plot first.")
            return

        try:
            file_name = input("\nEnter filename to save plot (e.g., plot.png): ").strip()
            if not file_name:
                print(" Skipped saving plot.")
                return

            self.last_plot.savefig(file_name)
            print(f" Plot saved successfully as '{file_name}'.")
        except Exception as e:
            print(f" Error saving plot: {e}")
        finally:
            if self.last_plot:
                plt.close(self.last_plot)
                print(" Figure closed and memory released.")
                self.last_plot = None


# --- Main Menu ---
def main():
    analyzer = DataAnalyzer()

    while True:
        print("\n====== MAIN MENU ======")
        print("1. Load Dataset")
        print("2. Explore Data")
        print("3. Perform DataFrame Operations")
        print("4. Clean and Handle Missing Data")
        print("5. Generate Descriptive Statistics")
        print("6. Data Visualization")
        print("7. Save Last Plot")
        print("8. Exit")

        choice = input("Enter your choice: ").strip()

        if choice == "1": analyzer.load_data()
        elif choice == "2": analyzer.explore()
        elif choice == "3":
            while True:
                print("\n--- OPERATIONS MENU ---")
                print("1. Convert to NumPy")
                print("2. Mathematical Operations")
                print("3. Combine Datasets")
                print("4. Split Dataset")
                print("5. Search / Sort / Filter")
                print("6. Aggregate Functions")
                print("7. Back to Main Menu")
                op = input("Enter operation: ").strip()
                
                if op == "1": analyzer.convert_numpy()
                elif op == "2": analyzer.mathematical_operations()
                elif op == "3": analyzer.combine_data()
                elif op == "4": analyzer.split_data()
                elif op == "5": analyzer.search_sort_filter()
                elif op == "6": analyzer.aggregate_functions()
                elif op == "7": break
                else: print("Invalid choice.")
        
        elif choice == "4": analyzer.clean_data()
        elif choice == "5":
            while True:
                print("\n--- STATISTICS MENU ---")
                print("1. Statistical Analysis")
                print("2. Pivot Table")
                print("3. Back to Main Menu")
                stat = input("Enter choice: ").strip()
                
                if stat == "1": analyzer.statistical_analysis()
                elif stat == "2": analyzer.create_pivot_table()
                elif stat == "3": break
                else: print("Invalid choice.")
                
        elif choice == "6": analyzer.visualize_data()
        elif choice == "7": analyzer.save_visualization()
        elif choice == "8":
            print(" Exiting the program. Goodbye!")
            break
        else:
            print("Invalid choice.")

if __name__ == "__main__":
    main()


#   Titanic.csv